# **Protein**

In [ ]:
# Step 1: Install and Setup
!pip uninstall -y torch torchvision torchaudio dgl
!pip install numpy==1.24.4
!pip install torch==2.1.0+cu118 torchvision==0.16.0+cu118 torchaudio==2.1.0+cu118 -f https://download.pytorch.org/whl/torch_stable.html
!pip install dgl -f https://data.dgl.ai/wheels/cu113/repo.html
!pip install torchdata==0.6.1
!pip install torch-geometric

# Step 2: Load Dataset as Dense
import torch
import torch.nn.functional as F
from torch_geometric.datasets import TUDataset
from torch_geometric.transforms import ToDense
from torch_geometric.loader import DenseDataLoader
from sklearn.model_selection import train_test_split

# Load dataset and get properties
dataset = TUDataset(root='/tmp/PROTEINS', name='PROTEINS')
num_features = dataset.num_node_features
num_classes = dataset.num_classes
max_nodes = max(data.num_nodes for data in dataset)

# Apply dense transform
dense_transform = ToDense(max_nodes)
dataset = [dense_transform(data) for data in dataset]

# Train/test split
train_dataset, test_dataset = train_test_split(dataset, test_size=0.2, stratify=[d.y.item() for d in dataset])
train_loader = DenseDataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DenseDataLoader(test_dataset, batch_size=32)

print(f"Dense data loaded — Features: {num_features}, Classes: {num_classes}, Max Nodes: {max_nodes}")


Looking in links: https://download.pytorch.org/whl/torch_stable.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 GB 678.4 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 78.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 MB 9.0 MB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 3.2.0
    Uninstalling triton-3.2.0:
      Successfully uninstalled triton-3.2.0
Looking in links: https://data.dgl.ai/wheels/cu113/repo.html
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.9/619.9 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━

Processing...
Done!


Dense data loaded — Features: 3, Classes: 2, Max Nodes: 620


In [ ]:
from math import ceil
from torch import nn
from torch_geometric.nn import DenseSAGEConv, dense_diff_pool

class GNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, lin=True):
        super().__init__()
        self.conv1 = DenseSAGEConv(in_channels, hidden_channels)
        self.bn1 = nn.BatchNorm1d(hidden_channels)
        self.conv2 = DenseSAGEConv(hidden_channels, hidden_channels)
        self.bn2 = nn.BatchNorm1d(hidden_channels)
        self.conv3 = DenseSAGEConv(hidden_channels, out_channels)
        self.bn3 = nn.BatchNorm1d(out_channels)
        self.lin = nn.Linear(2 * hidden_channels + out_channels, out_channels) if lin else None

    def bn(self, i, x):
        b, n, c = x.size()
        x = x.view(-1, c)
        x = getattr(self, f'bn{i}')(x)
        return x.view(b, n, c)

    def forward(self, x, adj):
        x1 = self.bn(1, self.conv1(x, adj).relu())
        x2 = self.bn(2, self.conv2(x1, adj).relu())
        x3 = self.bn(3, self.conv3(x2, adj).relu())
        x = torch.cat([x1, x2, x3], dim=-1)
        return self.lin(x).relu() if self.lin else x


In [ ]:
class DiffPool(nn.Module):
    def __init__(self, num_features, num_classes, max_nodes):
        super().__init__()
        num_nodes1 = ceil(0.25 * max_nodes)
        num_nodes2 = ceil(0.25 * num_nodes1)

        self.gnn1_pool = GNN(num_features, 64, num_nodes1)
        self.gnn1_embed = GNN(num_features, 64, 64, lin=False)

        self.gnn2_pool = GNN(3 * 64, 64, num_nodes2)
        self.gnn2_embed = GNN(3 * 64, 64, 64, lin=False)

        self.gnn3_embed = GNN(3 * 64, 64, 64, lin=False)

        self.lin1 = nn.Linear(3 * 64, 64)
        self.lin2 = nn.Linear(64, num_classes)

    def forward(self, x, adj):
        s = self.gnn1_pool(x, adj)
        x = self.gnn1_embed(x, adj)
        x, adj, l1, e1 = dense_diff_pool(x, adj, s)

        s = self.gnn2_pool(x, adj)
        x = self.gnn2_embed(x, adj)
        x, adj, l2, e2 = dense_diff_pool(x, adj, s)

        x = self.gnn3_embed(x, adj)
        x = x.mean(dim=1)
        x = self.lin1(x).relu()
        x = self.lin2(x)

        return F.log_softmax(x, dim=-1), l1 + l2, e1 + e2


In [ ]:
import torch
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize model and optimizer
model = DiffPool(num_features=num_features, num_classes=num_classes, max_nodes=max_nodes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training function
def train():
    model.train()
    total_loss, correct = 0, 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()

        # Forward pass
        out, link_loss, ent_loss = model(data.x, data.adj)

        # Compute loss (NLL + DiffPool regularization losses)
        loss = F.nll_loss(out, data.y.view(-1)) + link_loss + ent_loss
        loss.backward()
        optimizer.step()

        # Track loss and accuracy
        total_loss += loss.item() * data.y.size(0)
        correct += out.argmax(dim=1).eq(data.y.view(-1)).sum().item()

    avg_loss = total_loss / len(train_loader.dataset)
    acc = correct / len(train_loader.dataset)
    return avg_loss, acc

# Evaluation function
def test(loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out, _, _ = model(data.x, data.adj)
            correct += out.argmax(dim=1).eq(data.y.view(-1)).sum().item()
    return correct / len(loader.dataset)

# Training loop
for epoch in range(1, 1000):
    loss, train_acc = train()
    test_acc = test(test_loader)
    print(f"Epoch {epoch:02d} | Loss: {loss:.4f} | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")


Epoch 01 | Loss: 8.4670 | Train Acc: 0.7337 | Test Acc: 0.7130
Epoch 02 | Loss: 6.5281 | Train Acc: 0.7169 | Test Acc: 0.7758
Epoch 03 | Loss: 5.5487 | Train Acc: 0.7180 | Test Acc: 0.7713
Epoch 04 | Loss: 5.4241 | Train Acc: 0.7315 | Test Acc: 0.7399
Epoch 05 | Loss: 5.3773 | Train Acc: 0.7247 | Test Acc: 0.6951
Epoch 06 | Loss: 5.3327 | Train Acc: 0.7438 | Test Acc: 0.7758
Epoch 07 | Loss: 5.2845 | Train Acc: 0.7337 | Test Acc: 0.4619
Epoch 08 | Loss: 3.0532 | Train Acc: 0.7337 | Test Acc: 0.7803
Epoch 09 | Loss: 0.7679 | Train Acc: 0.6978 | Test Acc: 0.5964
Epoch 10 | Loss: 0.6064 | Train Acc: 0.7348 | Test Acc: 0.5964
Epoch 11 | Loss: 0.5749 | Train Acc: 0.7404 | Test Acc: 0.6188
Epoch 12 | Loss: 0.5795 | Train Acc: 0.7326 | Test Acc: 0.7220
Epoch 13 | Loss: 0.5628 | Train Acc: 0.7348 | Test Acc: 0.6009
Epoch 14 | Loss: 0.5478 | Train Acc: 0.7315 | Test Acc: 0.5964
Epoch 15 | Loss: 0.5464 | Train Acc: 0.7315 | Test Acc: 0.5964
Epoch 16 | Loss: 0.5407 | Train Acc: 0.7674 | Test Acc:

#Enzymes

In [ ]:
from torch_geometric.datasets import TUDataset
from torch_geometric.transforms import ToDense
from torch_geometric.loader import DenseDataLoader
from sklearn.model_selection import train_test_split

# Load ENZYMES dataset
dataset = TUDataset(root='/tmp/ENZYMES', name='ENZYMES')
num_features = dataset.num_node_features
num_classes = dataset.num_classes
max_nodes = max(data.num_nodes for data in dataset)

# Apply dense transform
dense_transform = ToDense(max_nodes)
dataset = [dense_transform(data) for data in dataset]

# Train/test split (stratified)
train_dataset, test_dataset = train_test_split(
    dataset, test_size=0.2, stratify=[d.y.item() for d in dataset], random_state=42
)

# Use DenseDataLoader
train_loader = DenseDataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DenseDataLoader(test_dataset, batch_size=32)

print(f"Dense ENZYMES dataset loaded — Features: {num_features}, Classes: {num_classes}, Max Nodes: {max_nodes}")


Dense ENZYMES dataset loaded — Features: 3, Classes: 6, Max Nodes: 126


In [ ]:
from math import ceil
from torch import nn
from torch_geometric.nn import DenseSAGEConv, dense_diff_pool

class GNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, lin=True):
        super().__init__()
        self.conv1 = DenseSAGEConv(in_channels, hidden_channels)
        self.bn1 = nn.BatchNorm1d(hidden_channels)
        self.conv2 = DenseSAGEConv(hidden_channels, hidden_channels)
        self.bn2 = nn.BatchNorm1d(hidden_channels)
        self.conv3 = DenseSAGEConv(hidden_channels, out_channels)
        self.bn3 = nn.BatchNorm1d(out_channels)
        self.lin = nn.Linear(2 * hidden_channels + out_channels, out_channels) if lin else None

    def bn(self, i, x):
        b, n, c = x.size()
        x = x.view(-1, c)
        x = getattr(self, f'bn{i}')(x)
        return x.view(b, n, c)

    def forward(self, x, adj):
        x1 = self.bn(1, self.conv1(x, adj).relu())
        x2 = self.bn(2, self.conv2(x1, adj).relu())
        x3 = self.bn(3, self.conv3(x2, adj).relu())
        x = torch.cat([x1, x2, x3], dim=-1)
        return self.lin(x).relu() if self.lin else x




class DiffPool(nn.Module):
    def __init__(self, num_features, num_classes, max_nodes):
        super().__init__()
        num_nodes1 = ceil(0.25 * max_nodes)
        num_nodes2 = ceil(0.25 * num_nodes1)

        self.gnn1_pool = GNN(num_features, 64, num_nodes1)
        self.gnn1_embed = GNN(num_features, 64, 64, lin=False)

        self.gnn2_pool = GNN(3 * 64, 64, num_nodes2)
        self.gnn2_embed = GNN(3 * 64, 64, 64, lin=False)

        self.gnn3_embed = GNN(3 * 64, 64, 64, lin=False)

        self.lin1 = nn.Linear(3 * 64, 64)
        self.lin2 = nn.Linear(64, num_classes)

    def forward(self, x, adj):
        s = self.gnn1_pool(x, adj)
        x = self.gnn1_embed(x, adj)
        x, adj, l1, e1 = dense_diff_pool(x, adj, s)

        s = self.gnn2_pool(x, adj)
        x = self.gnn2_embed(x, adj)
        x, adj, l2, e2 = dense_diff_pool(x, adj, s)

        x = self.gnn3_embed(x, adj)
        x = x.mean(dim=1)
        x = self.lin1(x).relu()
        x = self.lin2(x)

        return F.log_softmax(x, dim=-1), l1 + l2, e1 + e2



In [ ]:
import torch
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize model and optimizer
model = DiffPool(num_features=num_features, num_classes=num_classes, max_nodes=max_nodes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training function
def train():
    model.train()
    total_loss, correct = 0, 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()

        # Forward pass
        out, link_loss, ent_loss = model(data.x, data.adj)

        # Compute loss (NLL + DiffPool regularization losses)
        loss = F.nll_loss(out, data.y.view(-1)) + link_loss + ent_loss
        loss.backward()
        optimizer.step()

        # Track loss and accuracy
        total_loss += loss.item() * data.y.size(0)
        correct += out.argmax(dim=1).eq(data.y.view(-1)).sum().item()

    avg_loss = total_loss / len(train_loader.dataset)
    acc = correct / len(train_loader.dataset)
    return avg_loss, acc

# Evaluation function
def test(loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out, _, _ = model(data.x, data.adj)
            correct += out.argmax(dim=1).eq(data.y.view(-1)).sum().item()
    return correct / len(loader.dataset)

# Training loop
for epoch in range(1, 1000):
    loss, train_acc = train()
    test_acc = test(test_loader)
    print(f"Epoch {epoch:02d} | Loss: {loss:.4f} | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")


Epoch 01 | Loss: 6.5766 | Train Acc: 0.2292 | Test Acc: 0.2000
Epoch 02 | Loss: 5.4273 | Train Acc: 0.3104 | Test Acc: 0.1917
Epoch 03 | Loss: 4.5270 | Train Acc: 0.3625 | Test Acc: 0.3667
Epoch 04 | Loss: 3.6626 | Train Acc: 0.3417 | Test Acc: 0.3750
Epoch 05 | Loss: 3.0531 | Train Acc: 0.3875 | Test Acc: 0.3000
Epoch 06 | Loss: 2.4184 | Train Acc: 0.3958 | Test Acc: 0.4083
Epoch 07 | Loss: 1.9326 | Train Acc: 0.4229 | Test Acc: 0.3417
Epoch 08 | Loss: 1.7172 | Train Acc: 0.4417 | Test Acc: 0.3333
Epoch 09 | Loss: 1.6082 | Train Acc: 0.4542 | Test Acc: 0.3833
Epoch 10 | Loss: 1.6182 | Train Acc: 0.4479 | Test Acc: 0.3833
Epoch 11 | Loss: 1.5280 | Train Acc: 0.4708 | Test Acc: 0.3917
Epoch 12 | Loss: 1.5261 | Train Acc: 0.4667 | Test Acc: 0.3500
Epoch 13 | Loss: 1.5093 | Train Acc: 0.4750 | Test Acc: 0.4167
Epoch 14 | Loss: 1.5198 | Train Acc: 0.4604 | Test Acc: 0.3500
Epoch 15 | Loss: 1.4399 | Train Acc: 0.4833 | Test Acc: 0.3917
Epoch 16 | Loss: 1.4165 | Train Acc: 0.5167 | Test Acc: